# Bài 5: Phân Tích và Diễn Giải Mô Hình LightGBM

## Mục tiêu bài học

Sau bài học này, bạn sẽ có thể:
- Hiểu tại sao việc diễn giải mô hình (model interpretability) lại quan trọng trong các dự án khoa học dữ liệu.
- Trích xuất và trực quan hóa mức độ quan trọng của các đặc trưng (Feature Importance) từ mô hình LightGBM.
- Sử dụng thư viện `SHAP` để giải thích các dự đoán riêng lẻ và hiểu tác động của từng đặc trưng.
- Trực quan hóa các cây quyết định riêng lẻ trong mô hình LightGBM để hiểu cách nó đưa ra quyết định.

## 1. Tại sao phải Diễn giải Mô hình?

Các mô hình như LightGBM thường được coi là các "hộp đen" (black boxes). Chúng cho kết quả dự đoán rất chính xác, nhưng làm thế nào chúng đi đến kết luận đó thì không hề rõ ràng. Việc diễn giải mô hình là cực kỳ quan trọng vì nhiều lý do:

- **Xây dựng lòng tin:** Để các bên liên quan (sếp, khách hàng, người dùng cuối) tin tưởng và áp dụng mô hình, họ cần hiểu *tại sao* mô hình lại đưa ra quyết định như vậy.
- **Gỡ lỗi (Debugging):** Nếu mô hình đưa ra dự đoán lạ, việc hiểu được lý do sẽ giúp ta xác định các vấn đề tiềm ẩn trong dữ liệu hoặc logic của mô hình.
- **Khám phá tri thức:** Phân tích mô hình có thể giúp ta khám phá những mối quan hệ không ngờ tới trong dữ liệu, mang lại giá trị kinh doanh thực tế.
- **Đảm bảo công bằng và đạo đức:** Kiểm tra xem mô hình có đang dựa vào các đặc trưng nhạy cảm (như giới tính, chủng tộc) để đưa ra quyết định hay không, tránh các sai lệch (bias) không mong muốn.

## 2. Mức độ quan trọng của Đặc trưng (Feature Importance)

Đây là phương pháp đơn giản nhất để nhìn vào bên trong mô hình. LightGBM cung cấp sẵn các cách để đo lường mức độ quan trọng của từng đặc trưng.

- **`split`:** Số lần một đặc trưng được sử dụng để chia (split) trong các cây. Đặc trưng được dùng để chia nhiều hơn thì quan trọng hơn.
- **`gain`:** Tổng mức giảm của hàm mất mát (loss) khi sử dụng một đặc trưng để chia. Đây thường là chỉ số quan trọng nhất, vì nó cho biết đặc trưng đó đóng góp bao nhiêu vào việc cải thiện hiệu suất của mô hình.

LightGBM cung cấp hàm `lgb.plot_importance` để dễ dàng trực quan hóa điều này.

In [ ]:
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Tải và chuẩn bị dữ liệu (lấy từ bài trước)
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop(columns=['customerID'], inplace=True)

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].astype('category')

X = df.drop('Churn', axis=1)
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Huấn luyện một mô hình LightGBM đơn giản
model = lgb.LGBMClassifier(objective='binary', random_state=42)
model.fit(X_train, y_train, categorical_feature=list(categorical_cols))

In [ ]:
# Trực quan hóa Feature Importance
lgb.plot_importance(model, importance_type='gain', figsize=(10, 8), max_num_features=15)
plt.title('LightGBM Feature Importance (Gain)')
plt.show()

**Phân tích:**
Biểu đồ trên cho thấy `Contract`, `tenure` (thời gian khách hàng gắn bó), và `TotalCharges` là những yếu tố có ảnh hưởng lớn nhất đến quyết định của mô hình. Đây là một thông tin cực kỳ hữu ích cho bộ phận kinh doanh.

## 3. SHAP (SHapley Additive exPlanations): Giải thích từng dự đoán

Feature importance cho chúng ta cái nhìn tổng quan. Nhưng nếu sếp hỏi: "Tại sao khách hàng X lại được dự đoán là sẽ rời đi?", feature importance không thể trả lời được. Đây là lúc **SHAP** tỏa sáng.

SHAP dựa trên lý thuyết trò chơi để tính toán sự đóng góp của mỗi đặc trưng vào **một dự đoán cụ thể**. Nó trả lời câu hỏi: "Giá trị của đặc trưng A đã đẩy dự đoán của mô hình từ giá trị cơ sở (base value) lên/xuống bao nhiêu?"

- **Base Value:** Giá trị dự đoán trung bình trên toàn bộ tập dữ liệu.
- **SHAP Value:** Sự đóng góp của từng đặc trưng (có thể âm hoặc dương) vào việc đưa ra dự đoán cuối cùng cho một điểm dữ liệu.

Dự đoán cuối cùng = Base Value + Tổng các SHAP Value của tất cả các đặc trưng.

In [ ]:
# Cần cài đặt thư viện shap: pip install shap
import shap

# 1. Tạo một đối tượng explainer
explainer = shap.TreeExplainer(model)

# 2. Tính toán giá trị SHAP cho tập test
shap_values = explainer.shap_values(X_test)

# shap_values là một list gồm 2 mảng (cho lớp 0 và lớp 1)
# Chúng ta thường quan tâm đến lớp 1 (lớp 'Churn' trong trường hợp này)
shap_values_class1 = shap_values[1]

### a) Giải thích một dự đoán đơn lẻ (Local Interpretation)

Hãy chọn một khách hàng trong tập test được dự đoán là sẽ rời đi và xem tại sao.

In [ ]:
# Tìm một khách hàng được dự đoán là Churn (predicted_proba > 0.5)
y_pred_proba = model.predict_proba(X_test)[:, 1]
churn_customer_index = pd.Series(y_pred_proba).idxmax() # Lấy khách hàng có xác suất churn cao nhất

print(f"Phân tích khách hàng tại index: {churn_customer_index}")
print(f"Dữ liệu của khách hàng:\n{X_test.iloc[churn_customer_index]}")
print(f"\nXác suất Churn dự đoán: {y_pred_proba[churn_customer_index]:.4f}")

# Trực quan hóa giải thích cho khách hàng này
shap.initjs() # Khởi tạo javascript cho biểu đồ
shap.force_plot(explainer.expected_value[1], shap_values_class1[churn_customer_index, :], X_test.iloc[churn_customer_index, :])

**Cách đọc biểu đồ `force_plot`:**
- **`base value`**: Là xác suất churn trung bình trên toàn bộ tập dữ liệu.
- **Các mũi tên màu đỏ (features that push the prediction higher):** Là các đặc trưng đẩy dự đoán về phía churn (lớp 1). Giá trị của đặc trưng càng lớn, mũi tên càng dài. Ví dụ: `Contract=Month-to-month` là một yếu tố mạnh mẽ đẩy xác suất churn lên cao.
- **Các mũi tên màu xanh (features that push the prediction lower):** Là các đặc trưng kéo dự đoán về phía không churn (lớp 0). Ví dụ: `tenure` cao có thể là yếu tố giữ chân khách hàng.
- **`f(x)`**: Là xác suất churn dự đoán cuối cùng cho khách hàng này, sau khi tổng hợp tất cả các tác động.

### b) Giải thích toàn bộ mô hình (Global Interpretation)

SHAP cũng có thể tổng hợp các giải thích cục bộ để cho chúng ta một cái nhìn toàn cục, mạnh mẽ hơn cả feature importance truyền thống.

In [ ]:
# Biểu đồ tóm tắt SHAP
# Biểu đồ này cho thấy mức độ quan trọng của đặc trưng và tác động của chúng
shap.summary_plot(shap_values_class1, X_test, plot_type="dot")

**Cách đọc biểu đồ `summary_plot`:**
- **Thứ tự đặc trưng:** Các đặc trưng được sắp xếp theo mức độ quan trọng từ trên xuống dưới.
- **Mỗi điểm là một khách hàng:** Mỗi điểm trên biểu đồ đại diện cho một khách hàng trong tập dữ liệu.
- **Trục hoành (SHAP value):** Giá trị SHAP dương có nghĩa là đặc trưng đó làm tăng xác suất churn. Giá trị âm làm giảm xác suất churn.
- **Màu sắc (Feature value):** Màu đỏ biểu thị giá trị của đặc trưng đó cao, màu xanh biểu thị giá trị thấp.

**Phân tích từ biểu đồ trên:**
- **`Contract`:** Các giá trị cao (ví dụ: `Month-to-month` được mã hóa thành số lớn) có SHAP value dương, đẩy dự đoán về phía churn. Các giá trị thấp (hợp đồng dài hạn) có SHAP value âm, giữ chân khách hàng.
- **`tenure`:** Giá trị `tenure` thấp (màu xanh) có SHAP value dương (khách hàng mới dễ churn). Giá trị `tenure` cao (màu đỏ) có SHAP value âm (khách hàng lâu năm ít churn). Mối quan hệ này rất rõ ràng và hợp lý.

## 4. Trực quan hóa Cây (Tree Visualization)

Vì LightGBM là một mô hình dựa trên cây, chúng ta có thể trực quan hóa các cây quyết định riêng lẻ để hiểu cách nó hoạt động ở mức độ chi tiết nhất. Điều này hữu ích để hiểu các quy tắc rẽ nhánh đơn giản.

In [ ]:
lgb.plot_tree(model, tree_index=0, figsize=(20, 15), show_info=['split_gain'])
plt.show()

**Cách đọc biểu đồ cây:**
- Mỗi ô vuông là một nút (node).
- Dòng đầu tiên trong mỗi nút là điều kiện chia (ví dụ: `tenure <= 6.5`).
- `split_gain`: Mức độ lợi ích thu được khi thực hiện lần chia này.
- `output`: Giá trị dự đoán tại lá đó.
- `count`: Số lượng mẫu trong tập huấn luyện đi vào lá đó.

Việc phân tích một cây đơn lẻ giúp ta hiểu được các quy tắc quyết định ở mức độ vi mô, nhưng hãy nhớ rằng dự đoán cuối cùng là tổng hợp của hàng trăm cây như vậy.

## Tóm tắt

- Diễn giải mô hình là một bước không thể thiếu trong các dự án khoa học dữ liệu thực tế.
- **Feature Importance** cho ta cái nhìn tổng quan nhanh chóng về các yếu tố ảnh hưởng nhất.
- **SHAP** là một công cụ cực kỳ mạnh mẽ, cho phép giải thích cả dự đoán tổng thể (global) và dự đoán riêng lẻ (local), giúp trả lời câu hỏi "Tại sao?".
- **Trực quan hóa cây** giúp hiểu logic rẽ nhánh ở mức độ chi tiết nhất.

Chúc mừng bạn đã hoàn thành các bài học về LightGBM! Bây giờ là lúc áp dụng tất cả kiến thức đã học vào một dự án tổng hợp cuối khóa.